# Kaggle Inference Notebook

Notebook nay chay inference cho repo XLA tren Kaggle va tao `submission.csv` dung format cua challenge.


In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

KAGGLE_INPUT = Path('/kaggle/input')
WORKDIR = Path('/kaggle/working/XLA')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('Python:', sys.version)
print('Input datasets:')
for path in sorted(KAGGLE_INPUT.iterdir()):
    print('-', path)


In [ ]:
def find_repo_root(input_root: Path) -> Path:
    candidates = []
    for predict_file in input_root.rglob('predict.py'):
        root = predict_file.parent
        if (root / 'train.py').exists() and (root / 'utils').is_dir() and (root / 'models').is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Khong tim thay repo XLA trong /kaggle/input')
    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]

def find_local_checkpoint(repo_root: Path, input_root: Path) -> Path | None:
    direct = repo_root / 'models' / 'best.pth'
    if direct.exists():
        return direct
    matches = sorted(input_root.rglob('best.pth'))
    return matches[0] if matches else None

def find_test_image_dir(input_root: Path, repo_root: Path) -> Path:
    candidates = []
    repo_text = repo_root.as_posix().lower()
    for path in input_root.rglob('*'):
        if not path.is_dir():
            continue
        path_text = path.as_posix().lower()
        if path_text.startswith(repo_text):
            continue
        try:
            image_files = [p for p in path.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        except PermissionError:
            continue
        if not image_files:
            continue

        score = 0
        if 'test' in path_text:
            score += 100
        if 'hidden' in path_text:
            score += 20
        if 'private' in path_text:
            score += 10
        if 'image' in path_text:
            score += 5
        score += min(len(image_files), 10000) / 10000.0
        candidates.append((score, len(image_files), path))

    if not candidates:
        raise FileNotFoundError('Khong tim thay thu muc anh test trong /kaggle/input')

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]

REPO_SOURCE = find_repo_root(KAGGLE_INPUT)
CHECKPOINT_PATH = find_local_checkpoint(REPO_SOURCE, KAGGLE_INPUT)
TEST_IMAGE_DIR = find_test_image_dir(KAGGLE_INPUT, REPO_SOURCE)

print('REPO_SOURCE =', REPO_SOURCE)
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('TEST_IMAGE_DIR =', TEST_IMAGE_DIR)

assert REPO_SOURCE.exists()
assert TEST_IMAGE_DIR.exists()


Neu 3 path vua in ra chua dung, sua tay trong cell duoi day roi chay lai.


In [ ]:
# REPO_SOURCE = Path('/kaggle/input/ten-dataset-cua-ban/XLA')
# CHECKPOINT_PATH = Path('/kaggle/input/ten-model-dataset/best.pth')
# TEST_IMAGE_DIR = Path('/kaggle/input/ten-test-dataset/test/images')

print('REPO_SOURCE =', REPO_SOURCE)
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('TEST_IMAGE_DIR =', TEST_IMAGE_DIR)


In [ ]:
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
shutil.copytree(REPO_SOURCE, WORKDIR)
os.chdir(WORKDIR)

print('Current working directory:', Path.cwd())
print('Repo files:')
for path in sorted(Path.cwd().iterdir()):
    print('-', path.name)


In [ ]:
import torch
import yaml
import tqdm
from PIL import Image

print('Core dependencies are available')
print('Torch:', torch.__version__)


In [ ]:
predict_cmd = [
    sys.executable,
    'predict.py',
    '--image_dir', str(TEST_IMAGE_DIR),
    '--output', 'predictions.json',
    '--batch_size', '16',
]

if CHECKPOINT_PATH is not None:
    predict_cmd.extend(['--checkpoint', str(CHECKPOINT_PATH)])

print('Running command:')
print(' '.join(predict_cmd))
subprocess.run(predict_cmd, check=True)
print('Created predictions.json')


In [ ]:
with open('predictions.json', 'r', encoding='utf-8') as f:
    predictions = json.load(f)

print('Num predicted images:', len(predictions))
if predictions:
    print('First prediction item:')
    print(json.dumps(predictions[0], ensure_ascii=False, indent=2)[:1200])
else:
    print('Predictions file rong')


In [ ]:
def convert_predictions_to_submission_rows(predictions):
    rows = []
    for item in predictions:
        converted_boxes = []
        for box in item.get('boxes', []):
            x1, y1, x2, y2 = [float(v) for v in box['bbox']]
            converted_boxes.append({
                'x_min': round(x1, 4),
                'y_min': round(y1, 4),
                'x_max': round(x2, 4),
                'y_max': round(y2, 4),
                'class': str(box['class']),
                'confidence': round(float(box['confidence']), 6),
            })
        rows.append({
            'image_id': str(item['image_id']),
            'bounding_boxes': json.dumps(converted_boxes, ensure_ascii=False),
        })
    return rows

submission_rows = convert_predictions_to_submission_rows(predictions)
submission_df = pd.DataFrame(submission_rows, columns=['image_id', 'bounding_boxes'])
submission_df.to_csv('submission.csv', index=False, quoting=csv.QUOTE_MINIMAL)

print('Created submission.csv with shape:', submission_df.shape)
display(submission_df.head(10))


In [ ]:
assert Path('submission.csv').exists(), 'submission.csv chua duoc tao'
submission_df = pd.read_csv('submission.csv')
print(submission_df.shape)
print(submission_df.columns.tolist())
display(submission_df.head(5))

with open('submission.csv', 'r', encoding='utf-8') as f:
    for i in range(3):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())
